# Neural Network Hyperparameter Sweeps

This notebook runs W&B sweeps to find optimal hyperparameters for neural network models.

In [ ]:
# Only required on GPU Hub
%pip install dotenv wandb xgboost catboost lightning

In [1]:
import sys

sys.path.append("..")

import dotenv
import wandb

from src.api.run.neural_network import sweep_neural_network
from src.api.sweep import wandb_sweep
from src.models.neural_network.submission import create_submission

c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
c:\Users\kybur\Repos\HSLU\aicomp\code\.venv\Lib\site-packages\pydantic\_internal\_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because

In [2]:
dotenv.load_dotenv()
wandb.login()

wandb: Currently logged in as: v8-luky (aicomp-mmlm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Sweep 1: MSE as Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "NeuralNetwork: Architecture",
    "method": "bayes",
    "metric": {"name": "vali_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: MSE)

In [3]:
create_submission(season=2025, artifact_name="qu4h2dyd:v2")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='elu', dropout=0.3561403169684416, batch_norm=True, input_dropout=0.0, learning_rate=0.0002072894652897698, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.06311265923452011, momentum=0.9, scheduler=None, scheduler_step_size=30, scheduler_gamma=0.1, scheduler_patience=10, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=32, seed=42)", 'input_dim': 11, 'run_config': {'num_features': 11, 'start_season': 2003, 'valid_season': 2024}, 'neural_network_config': {'seed': 42, 'dropout': 0.3561403169684416, 'optimizer': 'adamw', 'activation': 'elu', 'batch_norm': True, 'weight_decay': 0.06311265923452011, 'hidden_layers': [64, 32], 'learning_rate': 0.0002072894652897698, 'loss_function': 'mse', 'early_stopping_patience': 10}}
Ru

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_2025.csv


## Sweep 2: BCE as Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (BCE)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "bce"},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 10, "max": 100},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: BCE)

In [4]:
create_submission(season=2025, artifact_name="b6t4o2d0:v1", submission_affix="bce")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[512, 256, 128, 64], activation='leaky_relu', dropout=0.4943348954859318, batch_norm=True, input_dropout=0.0, learning_rate=0.0002114486641540729, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.04722672654312865, momentum=0.9, scheduler=None, scheduler_step_size=30, scheduler_gamma=0.1, scheduler_patience=10, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='bce', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=32, seed=42)", 'input_dim': 10, 'run_config': {'num_features': 10, 'start_season': 2003, 'valid_season': 2024}, 'neural_network_config': {'seed': 42, 'dropout': 0.4943348954859318, 'optimizer': 'adamw', 'activation': 'leaky_relu', 'batch_norm': True, 'weight_decay': 0.04722672654312865, 'hidden_layers': [512, 256, 128, 64], 'learning_rate': 0.0002114486641540729, 'loss_function': 'bce'

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_bce_2025.csv


## Sweep 3: Default Features (num_features=0)

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (Default Features)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "mse"},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"value": 0},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Default Features)

In [5]:
create_submission(season=2025, artifact_name="6v4trgps:v1", submission_affix="default_features")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='gelu', dropout=0.4916954859372861, batch_norm=True, input_dropout=0.0, learning_rate=0.00012869467774969458, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.07535445370413052, momentum=0.9, scheduler=None, scheduler_step_size=30, scheduler_gamma=0.1, scheduler_patience=10, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='mse', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=32, seed=42)", 'input_dim': 29, 'run_config': {'num_features': 0, 'start_season': 2003, 'valid_season': 2024}, 'neural_network_config': {'seed': 42, 'dropout': 0.4916954859372861, 'optimizer': 'adamw', 'activation': 'gelu', 'batch_norm': True, 'weight_decay': 0.07535445370413052, 'hidden_layers': [64, 32], 'learning_rate': 0.00012869467774969458, 'loss_function': 'mse', 'early_stopping_patience': 10}}

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_default_features_2025.csv


## Sweep 4: BCE with Entropy Loss

In [ ]:
max_runs = 100
sweep_config = {
    "name": "Neural Network (BCE with Entropy Loss)",
    "method": "bayes",
    "metric": {"name": "val_brier", "goal": "minimize"},
    "parameters": {
        "neural_network_config": {
            "parameters": {
                "hidden_layers": {
                    "values": [
                        [64, 32],
                        [128, 64],
                        [128, 64, 32],
                        [256, 128, 64],
                        [256, 128, 64, 32],
                        [512, 256],
                        [512, 256, 128],
                        [512, 256, 128, 64],
                        [512, 256, 128, 64, 32],
                    ]
                },
                "activation": {"values": ["relu", "leaky_relu", "gelu", "elu"]},
                "dropout": {"distribution": "uniform", "min": 0.0, "max": 0.5},
                "batch_norm": {"value": True},
                "learning_rate": {"distribution": "log_uniform_values", "min": 0.0001, "max": 0.01},
                "optimizer": {"value": "adamw"},
                "weight_decay": {"distribution": "uniform", "min": 0.0, "max": 0.1},
                "early_stopping_patience": {"value": 10},
                "seed": {"value": 42},
                "loss_function": {"value": "bce_entropy"},
                "entropy_penalty_weight": {"distribution": "uniform", "min": 0.0, "max": 3.5},
            },
        },
        "run_config": {
            "parameters": {
                "valid_season": {"value": 2024},
                "start_season": {"value": 2003},
                "num_features": {"distribution": "int_uniform", "min": 5, "max": 87},
            },
        },
    },
}

In [ ]:
wandb_sweep(sweep_config, sweep_neural_network, run_count=max_runs, project="neural-network")

### Create Submission from Best Model (Loss: BCE with Entropy)

In [6]:
create_submission(season=2025, artifact_name="tfxfo799:v5", submission_affix="bce_entropy")

Config: {'config': "NeuralNetworkHyperparamConfig(hidden_layers=[64, 32], activation='leaky_relu', dropout=0.40085540627369354, batch_norm=True, input_dropout=0.0, learning_rate=0.0001055317154239594, batch_size=512, max_epochs=200, optimizer='adamw', weight_decay=0.09153865965611162, momentum=0.9, scheduler=None, scheduler_step_size=30, scheduler_gamma=0.1, scheduler_patience=10, early_stopping=True, early_stopping_patience=10, early_stopping_min_delta=0.0001, loss_function='bce_entropy', validation_split=0.1, weight_init='xavier_uniform', gradient_clip_val=None, accumulate_grad_batches=1, num_workers=16, seed=42)", 'input_dim': 10, 'run_config': {'num_features': 10, 'start_season': 2003, 'valid_season': 2024}, 'neural_network_config': {'seed': 42, 'dropout': 0.40085540627369354, 'optimizer': 'adamw', 'activation': 'leaky_relu', 'batch_norm': True, 'num_workers': 16, 'weight_decay': 0.09153865965611162, 'hidden_layers': [64, 32], 'learning_rate': 0.0001055317154239594, 'loss_function'

Seed set to 42
wandb:   1 of 1 files downloaded.  


Created submission at: C:\Users\kybur\Repos\HSLU\aicomp\code\submissions\submission_neural_network_bce_entropy_2025.csv
